In [207]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

from pandas.conftest import dropna

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [208]:
#import Libraries

import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [209]:
'''
#Run on Kaggle or Google Colab
df = pd.read_csv('/kaggle/input/datasets/mobeenfatimah/diabetes-risk-prediction-dataset-50k-patients/diabetes_risk_prediction_dataset.csv')
df
'''

"\n#Run on Kaggle or Google Colab\ndf = pd.read_csv('/kaggle/input/datasets/mobeenfatimah/diabetes-risk-prediction-dataset-50k-patients/diabetes_risk_prediction_dataset.csv')\ndf\n"

In [210]:
#Run it locally with you Kaggle API token
import kagglehub
kagglehub.login()

from pathlib import Path
path = kagglehub.dataset_download("mobeenfatimah/diabetes-risk-prediction-dataset-50k-patients")
csv_path = Path(path) / "diabetes_risk_prediction_dataset.csv"

df = pd.read_csv(csv_path)

In [211]:
df

,Patient_ID,Age,Gender,Country,Height_cm,Weight_kg,BMI,Waist_Circumference_cm,Blood_Glucose,HbA1c,...,Fatty_Liver,PCOS,Medication_Adherence,Work_Type,Residence_Type,Daily_Water_Intake_L,Diabetes_Risk_Score,AI_Health_Recommendation,Doctor_Consultation_Needed,Diabetes_Risk
0,1,32.0,Male,Mexico,182.1,65.8,19.8,71.2,88.4,10.4,...,Yes,No,Good,Retired,Rural,4.4,60,Healthy Diet Plan,Yes,Moderate
1,2,42.0,Other,Saudi Arabia,148.5,101.2,45.9,121.8,247.3,11.3,...,Yes,Yes,Average,Government,Rural,2.1,99,Begin Diabetes Management Plan,Yes,High
2,3,89.0,Other,Argentina,158.1,NaN,37.9,131.8,141.9,6.3,...,Yes,No,Average,Government,Urban,2.8,80,Strict Blood Sugar Monitoring,Yes,High
3,4,87.0,Other,United States,190.9,95.9,26.3,99.1,90.1,7.4,...,No,No,Average,Business,Rural,1.5,82,Begin Diabetes Management Plan,Yes,High
4,5,27.0,Other,Australia,156.9,101.9,41.4,77.1,93.8,12.0,...,Yes,Yes,Poor,Private,Rural,1.6,93,Begin Diabetes Management Plan,Yes,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,49996,52.0,Female,United Kingdom,153.9,107.4,45.3,75.9,248.8,9.9,...,No,Yes,Average,Retired,Urban,3.5,86,Consult Endocrinologist Immediately,Yes,High
49996,49997,53.0,Male,United Kingdom,179.6,64.0,19.8,114.5,198.4,6.3,...,Yes,No,Average,Government,Urban,3.4,49,Weight Management Program,No,Moderate
49997,49998,88.0,Male,Brazil,194.1,65.6,17.4,96.1,118.8,NaN,...,Yes,Yes,Poor,Retired,Urban,2.6,100,Begin Diabetes Management Plan,Yes,High
49998,49999,23.0,Other,Bangladesh,168.0,73.9,26.2,131.4,139.5,9.2,...,Yes,Yes,Good,Private,Urban,2.7,52,Increase Physical Activity,No,Moderate


In [212]:
df.shape


(50000, 41)

In [213]:
dataset_details = pd.DataFrame({
    "column": df.columns,
    "Data Type": [df[col].dtype for col in df.columns],
    "Missing Values": [f"{df[col].isna().mean() * 100:.1f}" for col in df.columns],
    "Unique Values": [df[col].nunique() for col in df.columns],
    "Example Value": [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns]
})
dataset_details

,column,Data Type,Missing Values,Unique Values,Example Value
0,Patient_ID,int64,0.0,50000,1
1,Age,float64,1.0,73,32.0
2,Gender,str,0.0,3,Male
3,Country,str,0.0,25,Mexico
4,Height_cm,float64,5.9,501,182.1
5,Weight_kg,float64,5.9,851,65.8
6,BMI,float64,0.0,497,19.8
7,Waist_Circumference_cm,float64,0.0,801,71.2
8,Blood_Glucose,float64,1.9,1801,88.4
9,HbA1c,float64,3.9,81,10.4


In [214]:
# Adjusting Missing Values

In [215]:
# Getting numeric columns with missing values
missing_values = df.select_dtypes(exclude='object').isnull().mean().round(2)
missing_values = missing_values[missing_values > 0].rename('missing_%')
list_missing_numeric_features = missing_values.index.tolist()
list_missing_numeric_features

['Age',
 'Height_cm',
 'Weight_kg',
 'Blood_Glucose',
 'HbA1c',
 'Total_Cholesterol',
 'HDL',
 'LDL',
 'Triglycerides',
 'Exercise_Hours_Per_Week',
 'Daily_Walking_Minutes',
 'Sleep_Hours']

In [216]:
# Getting object columns with missing values
missing_values = df.select_dtypes(include='object').isnull().mean().round(2)
missing_values = missing_values[missing_values > 0].rename('missing_%')
list_missing_object_features = missing_values.index.tolist()
list_missing_object_features

['Physical_Activity_Level', 'Medication_Adherence']

In [217]:
# Adjusting missing values in numeric Features
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('imputer', KNNImputer(n_neighbors=2))
])

df_statistics = df.copy()
df_corr = df.copy()

df_missing_adjustment = pipeline.fit_transform(df_corr[list_missing_numeric_features])
df_adjustment = pipeline.named_steps['scaler'].inverse_transform(df_missing_adjustment)
df_corr[list_missing_numeric_features] = df_adjustment

# Adjusting missing values in object features
df_corr[list_missing_object_features] = df_corr[list_missing_object_features].fillna(df_corr[list_missing_object_features].mode().iloc[0])


In [218]:
df_corr.isnull().mean().round(2)

Patient_ID                    0.0
Age                           0.0
Gender                        0.0
Country                       0.0
Height_cm                     0.0
Weight_kg                     0.0
BMI                           0.0
Waist_Circumference_cm        0.0
Blood_Glucose                 0.0
HbA1c                         0.0
Fasting_Blood_Sugar           0.0
Insulin_Level                 0.0
Blood_Pressure_Systolic       0.0
Blood_Pressure_Diastolic      0.0
Total_Cholesterol             0.0
HDL                           0.0
LDL                           0.0
Triglycerides                 0.0
Heart_Rate                    0.0
Physical_Activity_Level       0.0
Exercise_Hours_Per_Week       0.0
Daily_Walking_Minutes         0.0
Diet_Quality                  0.0
Sugar_Intake_Level            0.0
Sleep_Hours                   0.0
Stress_Level                  0.0
Smoking_Status                0.0
Alcohol_Consumption           0.0
Family_History_Diabetes       0.0
Hypertension  

In [219]:
df_corr[list_missing_numeric_features].describe().round(2)

,Age,Height_cm,Weight_kg,Blood_Glucose,HbA1c,Total_Cholesterol,HDL,LDL,Triglycerides,Exercise_Hours_Per_Week,Daily_Walking_Minutes,Sleep_Hours
count,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00
mean,53.88,170.00,87.46,159.90,8.49,220.14,57.37,135.04,229.56,5.02,89.95,6.50
std,21.07,14.25,24.13,51.79,2.29,57.51,18.63,48.88,97.76,2.85,51.51,2.01
min,18.00,145.00,45.00,70.00,4.50,120.00,25.00,50.00,60.00,0.00,0.00,3.00
25%,36.00,157.90,67.00,115.10,6.50,170.20,41.40,92.90,145.60,2.60,45.80,4.80
50%,54.00,169.95,87.40,159.90,8.50,220.30,57.30,135.20,229.50,5.00,89.90,6.50
75%,72.00,182.20,107.90,204.70,10.50,270.00,73.40,177.30,313.90,7.40,134.30,8.20
max,90.00,195.00,130.00,250.00,12.50,320.00,90.00,220.00,400.00,10.00,180.00,10.00


In [220]:
# Mapping features for further studies

In [221]:
#Mapping Booleans
boolean = ['Doctor_Consultation_Needed','Family_History_Diabetes','Hypertension','Heart_Disease','Fatty_Liver', 'PCOS']
df_corr[boolean] = df_corr[boolean].apply(lambda col: col.str.lower().map({'yes': 1, 'no': 0}))

#Mapping ordinal features
ordinal = ['Diabetes_Risk','Physical_Activity_Level','Sugar_Intake_Level','Stress_Level']
df_corr[ordinal] = df_corr[ordinal].apply(lambda col: col.str.lower().map({'low': 0, 'moderate': 1, 'high': 2}))

#Mapping others
df_corr['Medication_Adherence'] = df_corr['Medication_Adherence'].map({
    'Poor': 0,
    'Average': 1,
    'Good': 2
})

df_corr['Smoking_Status'] = df_corr['Smoking_Status'].map({
    'Never': 0,
    'Former': 1,
    'Current': 2
})

df_corr['Diet_Quality'] = df_corr['Diet_Quality'].map({
    'Poor': 0,
    'Average': 1,
    'Healthy': 2
})

df_corr['Gender'] = df_corr['Gender'].map({
    'Female': 0,
    'Male': 1,
    'Other': 2
})

df_corr['Alcohol_Consumption'] = df_corr['Alcohol_Consumption'].map({
    'Never': 0,
    'Occasionally': 1,
    'Frequently': 2
})

In [222]:
# Mapping Countries to Regions to avoid too many country columns

region_map = {
    # North America
    'United States': 'North America',
    'Canada': 'North America',
    'Mexico': 'North America',
    # South America
    'Brazil': 'South America',
    'Argentina': 'South America',
    # Europe
    'United Kingdom': 'Europe',
    'Germany': 'Europe',
    'France': 'Europe',
    'Spain': 'Europe',
    'Italy': 'Europe',
    'Russia': 'Europe',
    # Middle East
    'Saudi Arabia': 'Middle East',
    'Turkey': 'Middle East',
    'Egypt': 'Middle East',
    # South Asia
    'India': 'South Asia',
    'Pakistan': 'South Asia',
    'Bangladesh': 'South Asia',
    # East / Southeast Asia
    'China': 'East Asia',
    'Japan': 'East Asia',
    'South Korea': 'East Asia',
    'Indonesia': 'East Asia',
    'Malaysia': 'East Asia',
    # Africa
    'Nigeria': 'Africa',
    'South Africa': 'Africa',
    # Oceania
    'Australia': 'Oceania',
}

df_statistics['Region'] = df_statistics['Country'].map(region_map)
df_corr['Region'] = df_corr['Country'].map(region_map)

In [223]:
df_statistics.drop(['Country','Patient_ID','AI_Health_Recommendation'], axis=1, inplace=True)
df_corr.drop(['Country','Patient_ID','AI_Health_Recommendation'], axis=1, inplace=True)

In [224]:
# One-hot encoding for Work_Type and region to assess correlation
df_corr = pd.get_dummies(df_corr, columns=['Work_Type','Residence_Type','Region'],dtype=int)

In [225]:
# Filtering correlation matrix to assess multicolinearity
corr_matrix = df_corr.corr(method='spearman')
corr_filter_multi = corr_matrix[((corr_matrix >= 0.1) | (corr_matrix <= -0.1)) & (corr_matrix != 1.000)]
corr_filter_multi

,Age,Gender,Height_cm,Weight_kg,BMI,Waist_Circumference_cm,Blood_Glucose,HbA1c,Fasting_Blood_Sugar,Insulin_Level,...,Residence_Type_Rural,Residence_Type_Urban,Region_Africa,Region_East Asia,Region_Europe,Region_Middle East,Region_North America,Region_Oceania,Region_South America,Region_South Asia
Age,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Height_cm,NaN,NaN,NaN,NaN,-0.464651,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Weight_kg,NaN,NaN,NaN,NaN,0.823686,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BMI,NaN,NaN,-0.464651,0.823686,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Waist_Circumference_cm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Blood_Glucose,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HbA1c,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fasting_Blood_Sugar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Insulin_Level,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [226]:
# Filtering correlation matrix to assess correlations for the target variables
corr_matrix = df_corr.corr(method='spearman')
target_corr = corr_matrix[['Diabetes_Risk_Score','Doctor_Consultation_Needed']]
corr_matrix_target = target_corr[((target_corr >= 0.1) | (target_corr <= -0.1)) & (target_corr != 1.000)]
corr_matrix_target

,Diabetes_Risk_Score,Doctor_Consultation_Needed
Age,0.273961,0.109015
Gender,NaN,NaN
Height_cm,-0.100494,NaN
Weight_kg,0.195860,NaN
BMI,0.234544,NaN
Waist_Circumference_cm,NaN,NaN
Blood_Glucose,NaN,NaN
HbA1c,0.294889,0.129198
Fasting_Blood_Sugar,0.446435,0.186641
Insulin_Level,NaN,NaN


In [227]:
# Create individual dataframes for further ML models

# Extract feature names with non-NaN correlation for each target
features_diabetes_risk = corr_matrix_target['Diabetes_Risk_Score'].dropna().index.tolist()

# Dataframe to assess diabetes risk score
df_diabetes_risk = df_corr[features_diabetes_risk]
df_diabetes_risk.drop(['Diabetes_Risk','Doctor_Consultation_Needed'], axis=1, inplace=True)

# Dataframe to assess Doctor consultation needed
df_doctor_consultation = df_corr[['Doctor_Consultation_Needed','Diabetes_Risk_Score']]

In [228]:
df_diabetes_risk

,Age,Height_cm,Weight_kg,BMI,HbA1c,Fasting_Blood_Sugar,Blood_Pressure_Systolic,Blood_Pressure_Diastolic,Total_Cholesterol,Physical_Activity_Level,Diet_Quality,Stress_Level,Smoking_Status,Alcohol_Consumption,Family_History_Diabetes,Hypertension,Heart_Disease,Fatty_Liver,PCOS,Doctor_Consultation_Needed
0,32.0,182.1,65.8,19.8,10.40,149.5,94,61,138.7,1,2,1,1,0,1,0,1,1,0,1
1,42.0,148.5,101.2,45.9,11.30,199.3,148,100,286.8,2,2,1,2,2,1,0,1,1,1,1
2,89.0,158.1,84.5,37.9,6.30,219.6,101,108,129.4,2,1,2,0,2,1,0,0,1,0,1
3,87.0,190.9,95.9,26.3,7.40,217.7,120,63,168.2,2,1,2,2,1,1,1,0,0,0,1
4,27.0,156.9,101.9,41.4,12.00,153.5,190,90,281.5,1,0,0,0,1,0,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,52.0,153.9,107.4,45.3,9.90,191.5,97,94,193.1,0,0,0,1,1,1,0,0,0,1,1
49996,53.0,179.6,64.0,19.8,6.30,110.0,121,120,127.2,2,0,0,2,1,0,0,1,1,0,0
49997,88.0,194.1,65.6,17.4,8.35,208.4,176,119,137.6,1,0,1,0,2,1,1,0,1,1,1
49998,23.0,168.0,73.9,26.2,9.20,84.7,141,82,306.8,0,2,2,0,0,0,0,0,1,1,0


In [229]:
df_doctor_consultation

,Doctor_Consultation_Needed,Diabetes_Risk_Score
0,1,60
1,1,99
2,1,80
3,1,82
4,1,93
...,...,...
49995,1,86
49996,0,49
49997,1,100
49998,0,52
